Importando Bibliotecas

In [10]:
import pandas as pd
from google.cloud import bigquery
import requests
import time 

In [2]:
def consultar_ddd(ddd):
    """Consulta informações de um DDD na BrasilAPI"""
    url = f"https://brasilapi.com.br/api/ddd/v1/{ddd}"
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    else:
        return {"ddd": ddd, "erro": f"DDD não encontrado (status {response.status_code})"}

Exemplo: consultar DDD específico

In [3]:
ddd_sp = consultar_ddd(11)
print("DDD 11:")
print(ddd_sp)
print("\n" + "-"*50 + "\n")

DDD 11:
{'state': 'SP', 'cities': ['EMBU', 'VÁRZEA PAULISTA', 'VARGEM GRANDE PAULISTA', 'VARGEM', 'TUIUTI', 'TABOÃO DA SERRA', 'SUZANO', 'SÃO ROQUE', 'SÃO PAULO', 'SÃO LOURENÇO DA SERRA', 'SÃO CAETANO DO SUL', 'SÃO BERNARDO DO CAMPO', 'SANTO ANDRÉ', 'SANTANA DE PARNAÍBA', 'SANTA ISABEL', 'SALTO', 'SALESÓPOLIS', 'RIO GRANDE DA SERRA', 'RIBEIRÃO PIRES', 'POÁ', 'PIRAPORA DO BOM JESUS', 'PIRACAIA', 'PINHALZINHO', 'PEDRA BELA', 'OSASCO', 'NAZARÉ PAULISTA', 'MORUNGABA', 'MOGI DAS CRUZES', 'MAUÁ', 'MAIRIPORÃ', 'MAIRINQUE', 'JUQUITIBA', 'JUNDIAÍ', 'JOANÓPOLIS', 'JARINU', 'JANDIRA', 'ITUPEVA', 'ITU', 'ITATIBA', 'ITAQUAQUECETUBA', 'ITAPEVI', 'ITAPECERICA DA SERRA', 'IGARATÁ', 'GUARULHOS', 'GUARAREMA', 'FRANCO DA ROCHA', 'FRANCISCO MORATO', 'FERRAZ DE VASCONCELOS', 'EMBU-GUAÇU', 'DIADEMA', 'COTIA', 'CARAPICUÍBA', 'CAMPO LIMPO PAULISTA', 'CAJAMAR', 'CAIEIRAS', 'CABREÚVA', 'BRAGANÇA PAULISTA', 'BOM JESUS DOS PERDÕES', 'BIRITIBA-MIRIM', 'BARUERI', 'ATIBAIA', 'ARUJÁ', 'ARAÇARIGUAMA', 'ALUMÍNIO']}

--

Consultar múltiplos DDDs

In [4]:
lista_ddds = [11, 21, 31, 41, 51, 61, 71, 81, 91, 27]
resultados = []

for ddd in lista_ddds:
    dados = consultar_ddd(ddd)
    resultados.append(dados)
    print(f"✓ DDD {ddd} consultado")
    time.sleep(0.5)

✓ DDD 11 consultado
✓ DDD 21 consultado
✓ DDD 31 consultado
✓ DDD 41 consultado
✓ DDD 51 consultado
✓ DDD 61 consultado
✓ DDD 71 consultado
✓ DDD 81 consultado
✓ DDD 91 consultado
✓ DDD 27 consultado


Converter para DataFrame

In [5]:
df = pd.DataFrame(resultados)
print("\n" + "-"*50)
print("\nDataFrame com todos os resultados:")
print(df)


--------------------------------------------------

DataFrame com todos os resultados:
  state                                             cities
0    SP  [EMBU, VÁRZEA PAULISTA, VARGEM GRANDE PAULISTA...
1    RJ  [TERESÓPOLIS, TANGUÁ, SEROPÉDICA, SÃO JOÃO DE ...
2    MG  [QUELUZITA, VIÇOSA, VESPASIANO, URUCÂNIA, TIMÓ...
3    PR  [DOUTOR ULYSSES, TUNAS DO PARANÁ, TIJUCAS DO S...
4    RS  [SAO JOSE DO SUL, PÂNTANO GRANDE, XANGRI-LÁ, W...
5    DF  [BRASÍLIA, VILA BOA, VALPARAÍSO DE GOIÁS, SANT...
6    BA  [VERA CRUZ, SIMÕES FILHO, SAUBARA, SÃO SEBASTI...
7    PE  [ITAMARACÁ, XEXÉU, VITÓRIA DE SANTO ANTÃO, VIC...
8    PA  [SANTA ISABEL DO PARÁ, VISEU, VIGIA, ULIANÓPOL...
9    ES  [VITÓRIA, VILA VELHA, VILA VALÉRIO, VILA PAVÃO...


Mudando o estado do data frame, para ficar com uma coluna para estado e uma para cidade

In [6]:
df_explodido = df.explode('cities').reset_index(drop=True)

In [ ]:
df_explodido = df_explodido.rename(columns={
    'state': 'estado',
    'cities': 'cidade'
})

# Ordenando as colunas
df_explodido = df_explodido[['estado', 'cidade']]

Imprimindo o head do dataframe para verificação dos resultados.

In [9]:
print("Dataframe no formato desejado:")
print(df_explodido.head(10))
print(f"\nTotal de linhas: {len(df_explodido)}")

Dataframe no formato desejado:
  estado                  cidade
0     SP                    EMBU
1     SP         VÁRZEA PAULISTA
2     SP  VARGEM GRANDE PAULISTA
3     SP                  VARGEM
4     SP                  TUIUTI
5     SP         TABOÃO DA SERRA
6     SP                  SUZANO
7     SP               SÃO ROQUE
8     SP               SÃO PAULO
9     SP   SÃO LOURENÇO DA SERRA

Total de linhas: 675


Autenticação

In [11]:
client = bigquery.Client.from_service_account_json('C:/Users/ninem/Documents/storied-parser-332511-082bf009d3ff.json')

Definição da tabela do BigQuery

In [12]:
table =  "storied-parser-332511.General.DDDs"

Configurando o serviço de upload dos dados.

In [19]:
job_config = bigquery.LoadJobConfig(
    schema = [
        bigquery.SchemaField('estado', 'STRING'),
        bigquery.SchemaField('cidade', 'STRING')
    ],
    write_disposition = "WRITE_APPEND"
)

Carregando os dados para a tabela no BigQuery

In [20]:
load_job = client.load_table_from_dataframe(df_explodido, table, job_config=job_config)
load_job.result()

print(f"Tabela {table} criada com {len(df_explodido)} linhas.")

c:\Users\ninem\AppData\Local\Programs\Python\Python314\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Tabela storied-parser-332511.General.DDDs criada com 675 linhas.
